# NFL Data Analysis & Neural-Network Forecasting
## Position trends • next game • rest of season • next season

This is a **separate analysis notebook** designed to consume the versioned warehouse produced by `NFL_Public_Data_Warehouse.ipynb` plus the coaching add-on. It does not recollect the world; it loads the collected tables, creates leakage-aware features, evaluates baselines, and trains optional neural networks.

### Forecast targets
1. **Next game:** player and team performance for the next scheduled game.
2. **Rest of current season:** expected player/team outcomes over remaining games, conditional on the schedule and available information.
3. **Next year:** next-season performance. This is the most uncertain task because of roster turnover, coaching changes, injuries, draft picks, and role changes.

A model should never use future box-score data, post-game injury reports, final season statistics, or coach assignments not yet known at the forecast cutoff. The notebook uses chronological splits and compares neural nets to simple baselines. Install optional packages as needed.

In [1]:
from pathlib import Path
import json, warnings, math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

DATA_ROOT = Path("nfl_data")      # point to the warehouse export directory
COACH_ROOT = DATA_ROOT / "coaching"
OUTPUT_ROOT = Path("nfl_analysis")
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
RANDOM_STATE = 42
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 160)

def load_table(stem, roots=(DATA_ROOT, COACH_ROOT)):
    for root in roots:
        if not root.exists(): continue
        hits = list(root.rglob(f"{stem}.parquet")) + list(root.rglob(f"{stem}.csv"))
        if hits:
            p = hits[0]
            try:
                df = pd.read_parquet(p) if p.suffix == ".parquet" else pd.read_csv(p, low_memory=False)
                print(f"{stem}: {len(df):,} rows from {p}")
                return df
            except Exception as e:
                print(f"Failed reading {p}: {e}")
    print(f"Missing table: {stem}")
    return pd.DataFrame()

## 1. Load the warehouse and audit coverage

Update the table names if your collector used different names. Each DataFrame remains separate; this notebook creates derived analysis tables rather than replacing raw data.

In [2]:
TABLE_NAMES = [
    "pbp","schedules","player_stats","player_stats_weekly","team_stats","players","rosters",
    "weekly_rosters","snap_counts","ngs_passing","ngs_receiving","ngs_rushing","participation",
    "injuries","depth_charts","draft_picks","combine","pfr_passing","pfr_rushing","pfr_receiving",
    "team_game_tendencies","team_season_tendencies","coaches","coach_tenures","coaching_changes"
]
data = {name: load_table(name) for name in TABLE_NAMES}
coverage = pd.DataFrame([
    {"table": name, "rows": len(df), "columns": len(df.columns), "available": not df.empty}
    for name, df in data.items()
])
display(coverage)
coverage.to_csv(OUTPUT_ROOT / "input_coverage.csv", index=False)

pbp: 1,293,659 rows from nfl_data\pbp.parquet
schedules: 7,548 rows from nfl_data\schedules.parquet
player_stats: 481,734 rows from nfl_data\player_stats.parquet
Missing table: player_stats_weekly
team_stats: 14,691 rows from nfl_data\team_stats.parquet
players: 24,854 rows from nfl_data\players.parquet
rosters: 142,728 rows from nfl_data\rosters.parquet
weekly_rosters: 921,705 rows from nfl_data\weekly_rosters.parquet
snap_counts: 332,075 rows from nfl_data\snap_counts.parquet
Missing table: ngs_passing
Missing table: ngs_receiving
Missing table: ngs_rushing
participation: 478,989 rows from nfl_data\participation.parquet
injuries: 92,541 rows from nfl_data\injuries.parquet
depth_charts: 2,089,610 rows from nfl_data\depth_charts.parquet
draft_picks: 12,927 rows from nfl_data\draft_picks.parquet
combine: 8,968 rows from nfl_data\combine.parquet
Missing table: pfr_passing
Missing table: pfr_rushing
Missing table: pfr_receiving
team_game_tendencies: 22,125 rows from nfl_data\coaching\team

,table,rows,columns,available
0,pbp,1293659,372,True
1,schedules,7548,46,True
2,player_stats,481734,150,True
3,player_stats_weekly,0,0,False
4,team_stats,14691,138,True
5,players,24854,39,True
6,rosters,142728,36,True
7,weekly_rosters,921705,36,True
8,snap_counts,332075,16,True
9,ngs_passing,0,0,False


## 2. Identify player-performance data and normalize columns

`nflverse` releases differ in exact schemas. Inspect available columns before mapping. The next cell searches common weekly/player-stat table names and prints likely identifiers. Choose a single canonical source for each target to avoid duplicate player-game rows.

In [3]:
for name in ["player_stats","player_stats_weekly","pbp","rosters","weekly_rosters","snap_counts"]:
    df = data.get(name, pd.DataFrame())
    if not df.empty:
        print(f"\n{name} ({df.shape})")
        print(list(df.columns)[:100])


player_stats ((481734, 150))
['player_id', 'player_name', 'player_display_name', 'position', 'position_group', 'headshot_url', 'season', 'week', 'season_type', 'game_id', 'team', 'opponent_team', 'completions', 'attempts', 'passing_yards', 'passing_tds', 'passing_interceptions', 'sacks_suffered', 'sack_yards_lost', 'sack_fumbles', 'sack_fumbles_lost', 'passing_air_yards', 'passing_yards_after_catch', 'passing_first_downs', 'passing_epa', 'passing_cpoe', 'passing_2pt_conversions', 'pacr', 'passing_10', 'passing_16', 'passing_20', 'passing_40', 'carries', 'rushing_yards', 'rushing_tds', 'rushing_fumbles', 'rushing_fumbles_lost', 'rushing_first_downs', 'rushing_epa', 'rushing_2pt_conversions', 'rushing_10', 'rushing_12', 'rushing_20', 'rushing_40', 'receptions', 'targets', 'receiving_yards', 'receiving_tds', 'receiving_fumbles', 'receiving_fumbles_lost', 'receiving_air_yards', 'receiving_yards_after_catch', 'receiving_first_downs', 'receiving_epa', 'receiving_2pt_conversions', 'receiving

In [4]:
def choose_col(df, candidates):
    return next((c for c in candidates if c in df.columns), None)

def normalize_player_week(df, source_name="unknown"):
    if df.empty: return pd.DataFrame()
    x = df.copy()
    mapping = {
        "player_id": ["player_id","gsis_id","nfl_id","pfr_id"],
        "player_name": ["player_display_name","player_name","display_name","name"],
        "team": ["recent_team","team","posteam","club_code"],
        "position": ["position","position_group","pos"],
        "season": ["season","season_year"],
        "week": ["week","game_week"],
        "game_id": ["game_id","gsis_game_id"],
        "opponent": ["opponent_team","opponent","defteam"],
    }
    for canonical, candidates in mapping.items():
        col = choose_col(x, candidates)
        if col and col != canonical:
            x[canonical] = x[col]
    x["source_table"] = source_name
    for col in ["season","week"]:
        if col in x: x[col] = pd.to_numeric(x[col], errors="coerce")
    if "player_id" not in x and "player_name" not in x:
        return pd.DataFrame()
    return x

player_sources = []
for name in ["player_stats_weekly","player_stats"]:
    candidate = normalize_player_week(data.get(name, pd.DataFrame()), name)
    if not candidate.empty:
        player_sources.append(candidate)
if player_sources:
    player_games = pd.concat(player_sources, ignore_index=True, sort=False)
    dedup_keys = [c for c in ["player_id","season","week","game_id"] if c in player_games.columns]
    if len(dedup_keys) >= 3:
        player_games = player_games.drop_duplicates(dedup_keys, keep="first")
else:
    player_games = pd.DataFrame()
print("Canonical player_games:", player_games.shape)
display(player_games.head())

Canonical player_games: (481734, 152)


,player_id,player_name,player_display_name,position,position_group,headshot_url,season,week,season_type,game_id,team,opponent_team,completions,attempts,passing_yards,passing_tds,passing_interceptions,sacks_suffered,sack_yards_lost,sack_fumbles,sack_fumbles_lost,passing_air_yards,passing_yards_after_catch,passing_first_downs,passing_epa,passing_cpoe,passing_2pt_conversions,pacr,passing_10,passing_16,passing_20,passing_40,carries,rushing_yards,rushing_tds,rushing_fumbles,rushing_fumbles_lost,rushing_first_downs,rushing_epa,rushing_2pt_conversions,rushing_10,rushing_12,rushing_20,rushing_40,receptions,targets,receiving_yards,receiving_tds,receiving_fumbles,receiving_fumbles_lost,...,fg_att,fg_missed,fg_blocked,fg_long,fg_pct,fg_made_0_19,fg_made_20_29,fg_made_30_39,fg_made_40_49,fg_made_50_59,fg_made_60_,fg_missed_0_19,fg_missed_20_29,fg_missed_30_39,fg_missed_40_49,fg_missed_50_59,fg_missed_60_,fg_made_list,fg_missed_list,fg_blocked_list,fg_made_distance,fg_missed_distance,fg_blocked_distance,pat_made,pat_att,pat_missed,pat_blocked,pat_pct,gwfg_made,gwfg_att,gwfg_missed,gwfg_blocked,gwfg_distance,pt_att,pt_blocked,pt_long,pt_yards,pt_inside_20,pt_out_of_bounds,pt_downed,pt_touchback,pt_fair_caught,pt_returned,pt_return_yards,pt_return_tds,pt_net_yards,fantasy_points,fantasy_points_ppr,opponent,source_table
0,0,None,None,None,None,None,1999,1,REG,1999_01_MIN_ATL,ATL,MIN,0,0,0,0,0,0,0,0,0,0,0,0,NaN,NaN,0,NaN,0,0,0,0,0,0,0,0,0,0,NaN,0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,NaN,NaN,0,0,0,0,0,0,0,0,0,0,0,0,None,None,None,0,0,0,0,0,0,0,NaN,0,0,0,0,0,0,0,NaN,0,0,0,0,0,0,0,0,0,0,0.0,0.0,MIN,player_stats
1,00-0000003,Abdul-Karim al-Jabbar,Abdul-Karim al-Jabbar,RB,RB,https://static.www.nfl.com/image/private/f_aut...,1999,1,REG,1999_01_MIA_DEN,MIA,DEN,0,0,0,0,0,0,0,0,0,0,0,0,NaN,NaN,0,NaN,0,0,0,0,16,60,1,0,0,4,6.248771,0,2,1,0,0,1,1,7,0,0,0,...,0,0,0,NaN,NaN,0,0,0,0,0,0,0,0,0,0,0,0,None,None,None,0,0,0,0,0,0,0,NaN,0,0,0,0,0,0,0,NaN,0,0,0,0,0,0,0,0,0,0,12.7,13.7,DEN,player_stats
2,00-0000007,Rabih Abdullah,Rabih Abdullah,RB,RB,https://static.www.nfl.com/image/private/f_aut...,1999,1,REG,1999_01_NYG_TB,TB,NYG,0,0,0,0,0,0,0,0,0,0,0,0,NaN,NaN,0,NaN,0,0,0,0,1,0,0,0,0,0,-0.382897,0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,NaN,NaN,0,0,0,0,0,0,0,0,0,0,0,0,None,None,None,0,0,0,0,0,0,0,NaN,0,0,0,0,0,0,0,NaN,0,0,0,0,0,0,0,0,0,0,0.0,0.0,NYG,player_stats
3,00-0000008,Rahim Abdullah,Rahim Abdullah,OLB,LB,https://static.www.nfl.com/image/private/f_aut...,1999,1,REG,1999_01_PIT_CLE,CLE,PIT,0,0,0,0,0,0,0,0,0,0,0,0,NaN,NaN,0,NaN,0,0,0,0,0,0,0,0,0,0,NaN,0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,NaN,NaN,0,0,0,0,0,0,0,0,0,0,0,0,None,None,None,0,0,0,0,0,0,0,NaN,0,0,0,0,0,0,0,NaN,0,0,0,0,0,0,0,0,0,0,0.0,0.0,PIT,player_stats
4,00-0000017,Donnie Abraham,Donnie Abraham,CB,DB,https://static.www.nfl.com/image/private/f_aut...,1999,1,REG,1999_01_NYG_TB,TB,NYG,0,0,0,0,0,0,0,0,0,0,0,0,NaN,NaN,0,NaN,0,0,0,0,0,0,0,0,0,0,NaN,0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,NaN,NaN,0,0,0,0,0,0,0,0,0,0,0,0,None,None,None,0,0,0,0,0,0,0,NaN,0,0,0,0,0,0,0,NaN,0,0,0,0,0,0,0,0,0,0,0.0,0.0,NYG,player_stats


## 3. Position-level trends

Start with descriptive summaries. Use only seasons and columns that actually exist. For count outcomes, report sample size and distribution—not just means—and split by season, position, and role where possible.

In [5]:
if not player_games.empty:
    if "position" not in player_games:
        # Use a roster mapping when available; player IDs should be preferred over names.
        roster = data.get("weekly_rosters", pd.DataFrame())
        if roster.empty: roster = data.get("rosters", pd.DataFrame())
        if not roster.empty:
            rid = choose_col(roster, ["player_id","gsis_id","nfl_id"])
            rpos = choose_col(roster, ["position","position_group","pos"])
            if rid and rpos and "player_id" in player_games:
                player_games = player_games.merge(roster[[rid,rpos]].drop_duplicates(rid).rename(columns={rid:"player_id",rpos:"position"}), on="player_id", how="left")
    numeric_cols = [c for c in player_games.select_dtypes(include=np.number).columns
                    if c not in {"season","week","game_id","player_id"}]
    group_cols = [c for c in ["position","season"] if c in player_games.columns]
    if group_cols and numeric_cols:
        position_trends = player_games.groupby(group_cols, dropna=False)[numeric_cols].agg(["count","mean","median","std"]).reset_index()
        position_trends.columns = ["_".join(str(v) for v in col if str(v)!="").strip("_") if isinstance(col, tuple) else col for col in position_trends.columns]
        display(position_trends.head(30))
        position_trends.to_csv(OUTPUT_ROOT / "position_trends.csv", index=False)
    else:
        print("Could not find position and numeric outcome columns. Map the schema in the normalization cell.")
else:
    print("No player weekly/stat table found. Check coverage and source table names.")

,position,season,completions_count,completions_mean,completions_median,completions_std,attempts_count,attempts_mean,attempts_median,attempts_std,passing_yards_count,passing_yards_mean,passing_yards_median,passing_yards_std,passing_tds_count,passing_tds_mean,passing_tds_median,passing_tds_std,passing_interceptions_count,passing_interceptions_mean,passing_interceptions_median,passing_interceptions_std,sacks_suffered_count,sacks_suffered_mean,sacks_suffered_median,sacks_suffered_std,sack_yards_lost_count,sack_yards_lost_mean,sack_yards_lost_median,sack_yards_lost_std,sack_fumbles_count,sack_fumbles_mean,sack_fumbles_median,sack_fumbles_std,sack_fumbles_lost_count,sack_fumbles_lost_mean,sack_fumbles_lost_median,sack_fumbles_lost_std,passing_air_yards_count,passing_air_yards_mean,passing_air_yards_median,passing_air_yards_std,passing_yards_after_catch_count,passing_yards_after_catch_mean,passing_yards_after_catch_median,passing_yards_after_catch_std,passing_first_downs_count,passing_first_downs_mean,passing_first_downs_median,passing_first_downs_std,...,pt_long_median,pt_long_std,pt_yards_count,pt_yards_mean,pt_yards_median,pt_yards_std,pt_inside_20_count,pt_inside_20_mean,pt_inside_20_median,pt_inside_20_std,pt_out_of_bounds_count,pt_out_of_bounds_mean,pt_out_of_bounds_median,pt_out_of_bounds_std,pt_downed_count,pt_downed_mean,pt_downed_median,pt_downed_std,pt_touchback_count,pt_touchback_mean,pt_touchback_median,pt_touchback_std,pt_fair_caught_count,pt_fair_caught_mean,pt_fair_caught_median,pt_fair_caught_std,pt_returned_count,pt_returned_mean,pt_returned_median,pt_returned_std,pt_return_yards_count,pt_return_yards_mean,pt_return_yards_median,pt_return_yards_std,pt_return_tds_count,pt_return_tds_mean,pt_return_tds_median,pt_return_tds_std,pt_net_yards_count,pt_net_yards_mean,pt_net_yards_median,pt_net_yards_std,fantasy_points_count,fantasy_points_mean,fantasy_points_median,fantasy_points_std,fantasy_points_ppr_count,fantasy_points_ppr_mean,fantasy_points_ppr_median,fantasy_points_ppr_std
0,C,1999,187,0.0,0.0,0.0,187,0.000000,0.0,0.000000,187,0.0,0.0,0.0,187,0.0,0.0,0.0,187,0.0,0.0,0.0,187,0.0,0.0,0.0,187,0.0,0.0,0.0,187,0.0,0.0,0.0,187,0.0,0.0,0.0,187,0.0,0.0,0.0,187,0.0,0.0,0.0,187,0.0,0.0,0.0,...,NaN,NaN,187,0.0,0.0,0.0,187,0.0,0.0,0.0,187,0.0,0.0,0.0,187,0.0,0.0,0.0,187,0.0,0.0,0.0,187,0.0,0.0,0.0,187,0.0,0.0,0.0,187,0.0,0.0,0.0,187,0.0,0.0,0.0,187,0.0,0.0,0.0,187,0.000000,0.0,0.000000,187,0.000000,0.0,0.000000
1,C,2000,145,0.0,0.0,0.0,145,0.000000,0.0,0.000000,145,0.0,0.0,0.0,145,0.0,0.0,0.0,145,0.0,0.0,0.0,145,0.0,0.0,0.0,145,0.0,0.0,0.0,145,0.0,0.0,0.0,145,0.0,0.0,0.0,145,0.0,0.0,0.0,145,0.0,0.0,0.0,145,0.0,0.0,0.0,...,NaN,NaN,145,0.0,0.0,0.0,145,0.0,0.0,0.0,145,0.0,0.0,0.0,145,0.0,0.0,0.0,145,0.0,0.0,0.0,145,0.0,0.0,0.0,145,0.0,0.0,0.0,145,0.0,0.0,0.0,145,0.0,0.0,0.0,145,0.0,0.0,0.0,145,0.006207,0.0,0.074741,145,0.013103,0.0,0.157786
2,C,2001,140,0.0,0.0,0.0,140,0.000000,0.0,0.000000,140,0.0,0.0,0.0,140,0.0,0.0,0.0,140,0.0,0.0,0.0,140,0.0,0.0,0.0,140,0.0,0.0,0.0,140,0.0,0.0,0.0,140,0.0,0.0,0.0,140,0.0,0.0,0.0,140,0.0,0.0,0.0,140,0.0,0.0,0.0,...,NaN,NaN,140,0.0,0.0,0.0,140,0.0,0.0,0.0,140,0.0,0.0,0.0,140,0.0,0.0,0.0,140,0.0,0.0,0.0,140,0.0,0.0,0.0,140,0.0,0.0,0.0,140,0.0,0.0,0.0,140,0.0,0.0,0.0,140,0.0,0.0,0.0,140,0.045000,0.0,0.524003,140,0.052143,0.0,0.608509
3,C,2002,176,0.0,0.0,0.0,176,0.000000,0.0,0.000000,176,0.0,0.0,0.0,176,0.0,0.0,0.0,176,0.0,0.0,0.0,176,0.0,0.0,0.0,176,0.0,0.0,0.0,176,0.0,0.0,0.0,176,0.0,0.0,0.0,176,0.0,0.0,0.0,176,0.0,0.0,0.0,176,0.0,0.0,0.0,...,NaN,NaN,176,0.0,0.0,0.0,176,0.0,0.0,0.0,176,0.0,0.0,0.0,176,0.0,0.0,0.0,176,0.0,0.0,0.0,176,0.0,0.0,0.0,176,0.0,0.0,0.0,176,0.0,0.0,0.0,176,0.0,0.0,0.0,176,0.0,0.0,0.0,176,0.003977,0.0,0.052764,176,0.009659,0.0,0.128142
4,C,2003,232,0.0,0.0,0.0,232,0.000000,0.0,0.000000,232,0.0,0.0,0.0,232,0.0,0.0,0.0,232,0.0,0.0,0.0,232,0.0,0.0,0.0,232,0.0,0.0,0.0,232,0.0,0.0,0.0,232,0.0,0.0,0.0,232,0.0,0.0,0.0,232,0.0,0.0,0.0,232,0.0,0.0,0.0,...,NaN,NaN,232,0.0,0.0,0.0,2

## 4. Build a target

Pick one outcome that exists in the source table. This helper tries common fields; change `TARGET` after inspecting columns. Examples: passing yards, rushing yards, receiving yards, receptions, touchdowns, fantasy points, or team points. Predicting one target at a time is easier to validate than a single opaque multi-output model.

In [6]:
target_candidates = ["fantasy_points_ppr","fantasy_points","passing_yards","rushing_yards","receiving_yards",
                     "receptions","targets","carries","passing_tds","rushing_tds","receiving_tds","points"]
TARGET = next((c for c in target_candidates if c in player_games.columns), None) if not player_games.empty else None
print("Suggested target:", TARGET)
if TARGET:
    display(player_games[[c for c in ["player_name","player_id","position","team","season","week",TARGET] if c in player_games.columns]].head())
else:
    print("Set TARGET to a numeric outcome column after inspecting player_games.columns.")

Suggested target: fantasy_points_ppr


,player_name,player_id,position,team,season,week,fantasy_points_ppr
0,None,0,None,ATL,1999,1,0.0
1,Abdul-Karim al-Jabbar,00-0000003,RB,MIA,1999,1,13.7
2,Rabih Abdullah,00-0000007,RB,TB,1999,1,0.0
3,Rahim Abdullah,00-0000008,OLB,CLE,1999,1,0.0
4,Donnie Abraham,00-0000017,CB,TB,1999,1,0.0


## 5. Leakage-aware rolling features

The key rule: every feature for a game must be computed only from earlier games. We sort chronologically within player and shift before calculating rolling means. This simple feature builder supports a one-row-per-player-game table; if your source has multiple rows per player-game, deduplicate first.

In [7]:
def add_player_rolling_features(df, target, group_col="player_id", time_cols=("season","week"), windows=(3,5,8)):
    if target not in df: raise KeyError(f"Target {target!r} not found")
    x = df.copy()
    if group_col not in x:
        if "player_name" in x:
            group_col = "player_name"
        else:
            raise KeyError("Need player_id or player_name to create player history features")
    sort_cols = [c for c in [group_col, *time_cols] if c in x.columns]
    x = x.sort_values(sort_cols).reset_index(drop=True)
    # All rolling target features are lagged: no current-game target leakage.
    shifted = x.groupby(group_col, dropna=False)[target].shift(1)
    for w in windows:
        x[f"{target}_prior_mean_{w}"] = shifted.groupby(x[group_col], dropna=False).transform(
            lambda s: s.rolling(w, min_periods=1).mean()
        )
    x[f"{target}_prior_mean_all"] = x.groupby(group_col, dropna=False)[target].transform(
        lambda s: s.shift(1).expanding(min_periods=1).mean()
    )
    return x

model_df = pd.DataFrame()
if TARGET and not player_games.empty:
    model_df = add_player_rolling_features(player_games, TARGET)
    display(model_df[[c for c in ["player_name","position","season","week",TARGET] if c in model_df.columns] +
                     [c for c in model_df.columns if c.startswith(f"{TARGET}_prior_")]].head(15))

,player_name,position,season,week,fantasy_points_ppr,fantasy_points_ppr_prior_mean_3,fantasy_points_ppr_prior_mean_5,fantasy_points_ppr_prior_mean_8,fantasy_points_ppr_prior_mean_all
0,None,None,1999,1,0.0,NaN,NaN,NaN,NaN
1,None,None,1999,2,0.0,0.0,0.0,0.0,0.0
2,None,None,1999,3,0.0,0.0,0.0,0.0,0.0
3,None,None,1999,4,0.0,0.0,0.0,0.0,0.0
4,None,None,1999,5,0.0,0.0,0.0,0.0,0.0
5,None,None,1999,6,0.0,0.0,0.0,0.0,0.0
6,None,None,1999,7,0.0,0.0,0.0,0.0,0.0
7,None,None,1999,8,0.0,0.0,0.0,0.0,0.0
8,None,None,1999,9,0.0,0.0,0.0,0.0,0.0
9,None,None,1999,10,0.0,0.0,0.0,0.0,0.0


## 6. Chronological train/validation/test split

Do not randomly split games across time for forecasting. Train on earlier seasons, validate on a later season, and reserve the latest season(s) for a final untouched test. If there are too few seasons, use rolling-origin backtests.

In [8]:
def chronological_split(df, season_col="season", train_frac=0.70, valid_frac=0.15):
    if season_col not in df: raise KeyError(f"Missing {season_col}")
    seasons = sorted(pd.to_numeric(df[season_col], errors="coerce").dropna().unique())
    if len(seasons) < 3:
        raise ValueError(f"Need at least 3 distinct seasons for a season-based split; found {seasons}")
    n = len(seasons)
    train_end = max(1, int(n * train_frac))
    valid_end = max(train_end + 1, int(n * (train_frac + valid_frac)))
    valid_end = min(valid_end, n - 1)
    train_seasons = seasons[:train_end]
    valid_seasons = seasons[train_end:valid_end]
    test_seasons = seasons[valid_end:]
    return (df[df[season_col].isin(train_seasons)].copy(),
            df[df[season_col].isin(valid_seasons)].copy(),
            df[df[season_col].isin(test_seasons)].copy(),
            {"train":train_seasons,"validation":valid_seasons,"test":test_seasons})

if not model_df.empty and "season" in model_df:
    try:
        train_df, valid_df, test_df, split_info = chronological_split(model_df)
        print(split_info)
    except Exception as e:
        print("Split not ready:", e)

{'train': [1999, 2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017], 'validation': [2018, 2019, 2020, 2021], 'test': [2022, 2023, 2024, 2025, 2026]}


## 7. Baselines first

A neural network is not automatically better than a rolling average or a regularized linear model. Use MAE/RMSE for continuous outcomes and compare every neural network against at least a mean/rolling baseline. The target row must have a valid prior feature, and missing values should be handled within the training pipeline.

In [9]:
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import Ridge
from sklearn.dummy import DummyRegressor

def evaluate_regression(y_true, y_pred, label):
    return {"model":label, "n":len(y_true), "MAE":mean_absolute_error(y_true,y_pred),
            "RMSE":float(np.sqrt(mean_squared_error(y_true,y_pred)))}

baseline_results = pd.DataFrame()
feature_cols = [c for c in model_df.columns if c.startswith(f"{TARGET}_prior_")] if TARGET and not model_df.empty else []
if not model_df.empty and TARGET and feature_cols and "season" in model_df:
    valid_rows = model_df[TARGET].notna()
    model_df = model_df.loc[valid_rows].copy()
    # Recreate split after dropping missing target rows.
    try:
        train_df, valid_df, test_df, split_info = chronological_split(model_df)
        train_df = train_df.dropna(subset=feature_cols)
        test_df = test_df.dropna(subset=[TARGET])
        X_train, y_train = train_df[feature_cols], train_df[TARGET]
        X_test, y_test = test_df[feature_cols], test_df[TARGET]
        if len(train_df) and len(test_df):
            ridge = Pipeline([("imputer",SimpleImputer(strategy="median")),
                              ("scale",StandardScaler()),("model",Ridge(alpha=10.0))])
            ridge.fit(X_train,y_train)
            ridge_pred = ridge.predict(X_test)
            naive_pred = np.repeat(float(y_train.mean()), len(y_test))
            baseline_results = pd.DataFrame([
                evaluate_regression(y_test, naive_pred, "training-mean baseline"),
                evaluate_regression(y_test, ridge_pred, "Ridge rolling-history baseline"),
            ])
            display(baseline_results)
            baseline_results.to_csv(OUTPUT_ROOT / "baseline_metrics.csv", index=False)
    except Exception as e:
        print("Baseline evaluation not ready:", repr(e))

,model,n,MAE,RMSE
0,training-mean baseline,81454,3.788135,5.794350
1,Ridge rolling-history baseline,81454,1.561129,3.535626


## 8. Optional neural network (MLP)

This is a compact feed-forward baseline for tabular features, not a temporal transformer. Start with player-history features, then carefully add pregame availability, opponent defense, venue, rest days, weather, team pace, coaching tenure/tendencies, and depth-chart role. Every added feature needs a timestamped availability rule.

In [10]:
NN_AVAILABLE = True
try:
    import tensorflow as tf
    from tensorflow import keras
except Exception:
    NN_AVAILABLE = False
    print("TensorFlow is not installed. To enable the optional MLP, install tensorflow in your environment.")

nn_metrics = pd.DataFrame()
if NN_AVAILABLE and not model_df.empty and TARGET and feature_cols and "season" in model_df:
    try:
        # Reuse chronological split, with imputation/scaling fitted only on training data.
        train_df, valid_df, test_df, split_info = chronological_split(model_df)
        train_df = train_df.dropna(subset=[TARGET] + feature_cols)
        valid_df = valid_df.dropna(subset=[TARGET] + feature_cols)
        test_df = test_df.dropna(subset=[TARGET] + feature_cols)
        if min(len(train_df), len(valid_df), len(test_df)) > 0:
            imputer = SimpleImputer(strategy="median")
            scaler = StandardScaler()
            Xtr = scaler.fit_transform(imputer.fit_transform(train_df[feature_cols]))
            Xva = scaler.transform(imputer.transform(valid_df[feature_cols]))
            Xte = scaler.transform(imputer.transform(test_df[feature_cols]))
            ytr = train_df[TARGET].astype(float).to_numpy()
            yva = valid_df[TARGET].astype(float).to_numpy()
            yte = test_df[TARGET].astype(float).to_numpy()
            tf.keras.utils.set_random_seed(RANDOM_STATE)
            model = keras.Sequential([
                keras.layers.Input(shape=(Xtr.shape[1],)),
                keras.layers.Dense(64, activation="relu"),
                keras.layers.Dropout(0.20),
                keras.layers.Dense(32, activation="relu"),
                keras.layers.Dense(1)
            ])
            model.compile(optimizer=keras.optimizers.Adam(learning_rate=1e-3), loss="mse", metrics=["mae"])
            early_stop = keras.callbacks.EarlyStopping(monitor="val_loss", patience=8, restore_best_weights=True)
            model.fit(Xtr, ytr, validation_data=(Xva,yva), epochs=100, batch_size=128,
                      callbacks=[early_stop], verbose=1)
            pred = model.predict(Xte, verbose=0).ravel()
            nn_metrics = pd.DataFrame([evaluate_regression(yte,pred,"MLP neural network")])
            display(nn_metrics)
            nn_metrics.to_csv(OUTPUT_ROOT / "neural_network_metrics.csv", index=False)
            model.save(OUTPUT_ROOT / "player_outcome_mlp.keras")
    except Exception as e:
        print("Neural network run failed; inspect target/feature types and split sizes:", repr(e))

Epoch 1/100
2506/2506 [==============================] - 12s 4ms/step - loss: 13.6306 - mae: 1.5834 - val_loss: 14.0343 - val_mae: 1.6735
Epoch 2/100
2506/2506 [==============================] - 9s 4ms/step - loss: 13.0637 - mae: 1.5502 - val_loss: 14.0187 - val_mae: 1.6664
Epoch 3/100
2506/2506 [==============================] - 10s 4ms/step - loss: 13.0380 - mae: 1.5461 - val_loss: 14.0912 - val_mae: 1.6199
Epoch 4/100
2506/2506 [==============================] - 8s 3ms/step - loss: 13.0336 - mae: 1.5441 - val_loss: 14.0009 - val_mae: 1.6222
Epoch 5/100
2506/2506 [==============================] - 9s 4ms/step - loss: 13.0118 - mae: 1.5422 - val_loss: 14.0103 - val_mae: 1.6291
Epoch 6/100
2506/2506 [==============================] - 9s 4ms/step - loss: 12.9991 - mae: 1.5411 - val_loss: 14.0082 - val_mae: 1.6006
Epoch 7/100
2506/2506 [==============================] - 10s 4ms/step - loss: 12.9681 - mae: 1.5397 - val_loss: 14.1794 - val_mae: 1.6033
Epoch 8/100
2506/2506 [===============

,model,n,MAE,RMSE
0,MLP neural network,79698,1.516609,3.533194


## 9. Next-game predictions

For a real next-game forecast, create one row per player expected to participate in the upcoming game and compute the same lagged features from completed games only. Add schedule and pregame context, then train on historical rows using the same cutoff logic. The following helper selects the latest observed player row as a **starter template**, not a fully schedule-aware production forecast.

In [11]:
def latest_player_feature_rows(df, group_col="player_id", season_col="season", week_col="week"):
    if df.empty: return df.copy()
    if group_col not in df and "player_name" in df: group_col = "player_name"
    sort_cols = [c for c in [group_col, season_col, week_col] if c in df.columns]
    return df.sort_values(sort_cols).groupby(group_col, dropna=False).tail(1).copy()

if not model_df.empty:
    latest_rows = latest_player_feature_rows(model_df)
    display(latest_rows.head())
    latest_rows.to_csv(OUTPUT_ROOT / "latest_player_feature_rows_template.csv", index=False)

,player_id,player_name,player_display_name,position,position_group,headshot_url,season,week,season_type,game_id,team,opponent_team,completions,attempts,passing_yards,passing_tds,passing_interceptions,sacks_suffered,sack_yards_lost,sack_fumbles,sack_fumbles_lost,passing_air_yards,passing_yards_after_catch,passing_first_downs,passing_epa,passing_cpoe,passing_2pt_conversions,pacr,passing_10,passing_16,passing_20,passing_40,carries,rushing_yards,rushing_tds,rushing_fumbles,rushing_fumbles_lost,rushing_first_downs,rushing_epa,rushing_2pt_conversions,rushing_10,rushing_12,rushing_20,rushing_40,receptions,targets,receiving_yards,receiving_tds,receiving_fumbles,receiving_fumbles_lost,...,fg_pct,fg_made_0_19,fg_made_20_29,fg_made_30_39,fg_made_40_49,fg_made_50_59,fg_made_60_,fg_missed_0_19,fg_missed_20_29,fg_missed_30_39,fg_missed_40_49,fg_missed_50_59,fg_missed_60_,fg_made_list,fg_missed_list,fg_blocked_list,fg_made_distance,fg_missed_distance,fg_blocked_distance,pat_made,pat_att,pat_missed,pat_blocked,pat_pct,gwfg_made,gwfg_att,gwfg_missed,gwfg_blocked,gwfg_distance,pt_att,pt_blocked,pt_long,pt_yards,pt_inside_20,pt_out_of_bounds,pt_downed,pt_touchback,pt_fair_caught,pt_returned,pt_return_yards,pt_return_tds,pt_net_yards,fantasy_points,fantasy_points_ppr,opponent,source_table,fantasy_points_ppr_prior_mean_3,fantasy_points_ppr_prior_mean_5,fantasy_points_ppr_prior_mean_8,fantasy_points_ppr_prior_mean_all
41,0,None,None,None,None,None,2000,21,POST,2000_21_BAL_NYG,NYG,BAL,0,0,0,0,0,0,0,0,0,0,0,0,NaN,NaN,0,NaN,0,0,0,0,0,0,0,0,0,0,NaN,0,0,0,0,0,0,0,0,0,0,0,...,NaN,0,0,0,0,0,0,0,0,0,0,0,0,None,None,None,0,0,0,0,0,0,0,NaN,0,0,0,0,0,0,0,NaN,0,0,0,0,0,0,0,0,0,0,0.0,0.0,BAL,player_stats,0.000000,0.00,0.0000,0.000000
55,00-0000003,Abdul-Karim al-Jabbar,Abdul-Karim al-Jabbar,RB,RB,https://static.www.nfl.com/image/private/f_aut...,2000,1,REG,2000_01_IND_KC,IND,KC,0,0,0,0,0,0,0,0,0,0,0,0,NaN,NaN,0,NaN,0,0,0,0,1,-2,0,0,0,0,-0.549944,0,0,0,0,0,0,0,0,0,0,0,...,NaN,0,0,0,0,0,0,0,0,0,0,0,0,None,None,None,0,0,0,0,0,0,0,NaN,0,0,0,0,0,0,0,NaN,0,0,0,0,0,0,0,0,0,0,-0.2,-0.2,KC,player_stats,8.600000,7.28,6.3125,6.300000
127,00-0000007,Rabih Abdullah,Rabih Abdullah,RB,RB,https://static.www.nfl.com/image/private/f_aut...,2004,21,POST,2004_21_NE_PHI,NE,PHI,0,0,0,0,0,0,0,0,0,0,0,0,NaN,NaN,0,NaN,0,0,0,0,0,0,0,0,0,0,NaN,0,0,0,0,0,0,0,0,0,0,0,...,NaN,0,0,0,0,0,0,0,0,0,0,0,0,None,None,None,0,0,0,0,0,0,0,NaN,0,0,0,0,0,0,0,NaN,0,0,0,0,0,0,0,0,0,0,0.0,0.0,PHI,player_stats,0.333333,0.10,1.1500,0.700000
154,00-0000008,Rahim Abdullah,Rahim Abdullah,OLB,LB,https://static.www.nfl.com/image/private/f_aut...,2000,16,REG,2000_16_TEN_CLE,CLE,TEN,0,0,0,0,0,0,0,0,0,0,0,0,NaN,NaN,0,NaN,0,0,0,0,0,0,0,0,0,0,NaN,0,0,0,0,0,0,0,0,0,0,0,...,NaN,0,0,0,0,0,0,0,0,0,0,0,0,None,None,None,0,0,0,0,0,0,0,NaN,0,0,0,0,0,0,0,NaN,0,0,0,0,0,0,0,0,0,0,0.0,0.0,TEN,player_stats,0.000000,0.00,0.0000,0.000000
246,00-0000017,Donnie Abraham,Donnie Abraham,CB,DB,https://static.www.nfl.com/image/private/f_aut...,2004,19,POST,2004_19_NYJ_PIT,NYJ,PIT,0,0,0,0,0,0,0,0,0,0,0,0,NaN,NaN,0,NaN,0,0,0,0,0,0,0,0,0,0,NaN,0,0,0,0,0,0,0,0,0,0,0,...,NaN,0,0,0,0,0,0,0,0,0,0,0,0,None,None,None,0,0,0,0,0,0,0,NaN,0,0,0,0,0,0,0,NaN,0,0,0,0,0,0,0,0,0,0,0.0,0.0,PIT,player_stats,0.000000,0.00,0.0000,0.065934


## 10. Rest-of-season forecasts

A defensible rest-of-season projection is a **game-by-game simulation**:
- enumerate remaining schedule games;
- predict player opportunity and per-game outcomes conditional on current role, opponent, venue, and availability;
- simulate outcomes and sum only future games;
- update inputs after each newly completed game;
- report median and prediction intervals, not a single false-precision number.

Do not simply multiply a season-to-date average by remaining games when player role, injuries, bye weeks, and schedule vary. The code below creates a schedule audit template if a schedules table is present.

In [12]:
schedule = data.get("schedules", pd.DataFrame())
if not schedule.empty:
    print("Schedule columns:", list(schedule.columns))
    display(schedule.head())
    date_col = choose_col(schedule, ["gameday","game_date","date"])
    home_col = choose_col(schedule, ["home_team"])
    away_col = choose_col(schedule, ["away_team"])
    week_col = choose_col(schedule, ["week"])
    season_col = choose_col(schedule, ["season"])
    schedule_template = schedule.copy()
    schedule_template["forecast_status"] = "needs_cutoff_and_model"
    schedule_template.to_csv(OUTPUT_ROOT / "schedule_forecast_template.csv", index=False)

Schedule columns: ['game_id', 'season', 'game_type', 'week', 'gameday', 'weekday', 'gametime', 'away_team', 'away_score', 'home_team', 'home_score', 'location', 'result', 'total', 'overtime', 'old_game_id', 'gsis', 'nfl_detail_id', 'pfr', 'pff', 'espn', 'ftn', 'away_rest', 'home_rest', 'away_moneyline', 'home_moneyline', 'spread_line', 'away_spread_odds', 'home_spread_odds', 'total_line', 'under_odds', 'over_odds', 'div_game', 'roof', 'surface', 'temp', 'wind', 'away_qb_id', 'home_qb_id', 'away_qb_name', 'home_qb_name', 'away_coach', 'home_coach', 'referee', 'stadium_id', 'stadium']


,game_id,season,game_type,week,gameday,weekday,gametime,away_team,away_score,home_team,home_score,location,result,total,overtime,old_game_id,gsis,nfl_detail_id,pfr,pff,espn,ftn,away_rest,home_rest,away_moneyline,home_moneyline,spread_line,away_spread_odds,home_spread_odds,total_line,under_odds,over_odds,div_game,roof,surface,temp,wind,away_qb_id,home_qb_id,away_qb_name,home_qb_name,away_coach,home_coach,referee,stadium_id,stadium
0,1999_01_MIN_ATL,1999,REG,1,1999-09-12,Sunday,None,MIN,17.0,ATL,14.0,Home,-3.0,31.0,0.0,1999091210,598.0,None,199909120atl,NaN,190912001,NaN,7,7,NaN,NaN,-4.0,NaN,NaN,49.0,NaN,NaN,0,dome,astroturf,NaN,NaN,00-0003761,00-0002876,Randall Cunningham,Chris Chandler,Dennis Green,Dan Reeves,Gerry Austin,ATL00,Georgia Dome
1,1999_01_KC_CHI,1999,REG,1,1999-09-12,Sunday,None,KC,17.0,CHI,20.0,Home,3.0,37.0,0.0,1999091206,597.0,None,199909120chi,NaN,190912003,NaN,7,7,NaN,NaN,-3.0,NaN,NaN,38.0,NaN,NaN,0,outdoors,grass,80.0,12.0,00-0006300,00-0010560,Elvis Grbac,Shane Matthews,Gunther Cunningham,Dick Jauron,Phil Luckett,CHI98,Soldier Field
2,1999_01_PIT_CLE,1999,REG,1,1999-09-12,Sunday,None,PIT,43.0,CLE,0.0,Home,-43.0,43.0,0.0,1999091213,604.0,None,199909120cle,NaN,190912005,NaN,7,7,NaN,NaN,-6.0,NaN,NaN,37.0,NaN,NaN,1,outdoors,grass,78.0,12.0,00-0015700,00-0004230,Kordell Stewart,Ty Detmer,Bill Cowher,Chris Palmer,Bob McElwee,CLE00,Cleveland Browns Stadium
3,1999_01_OAK_GB,1999,REG,1,1999-09-12,Sunday,None,OAK,24.0,GB,28.0,Home,4.0,52.0,0.0,1999091208,602.0,None,199909120gnb,NaN,190912009,NaN,7,7,NaN,NaN,9.0,NaN,NaN,43.0,NaN,NaN,0,outdoors,grass,67.0,10.0,00-0005741,00-0005106,Rich Gannon,Brett Favre,Jon Gruden,Ray Rhodes,Tony Corrente,GNB00,Lambeau Field
4,1999_01_BUF_IND,1999,REG,1,1999-09-12,Sunday,None,BUF,14.0,IND,31.0,Home,17.0,45.0,0.0,1999091202,591.0,None,199909120clt,NaN,190912011,NaN,7,7,NaN,NaN,-3.0,NaN,NaN,45.5,NaN,NaN,1,dome,astroturf,NaN,NaN,00-0005363,00-0010346,Doug Flutie,Peyton Manning,Wade Phillips,Jim Mora,Ron Blum,IND99,RCA Dome


## 11. Next-year forecasts

Next-year prediction should be treated as a separate model/task:
- predict role/opportunity (roster retention, depth-chart rank, snaps, targets/carries);
- predict per-opportunity efficiency and production;
- include age, prior-year performance, career trajectory, draft capital, coaching/team changes, and roster competition when available;
- account for rookies with limited NFL history using college data and draft information;
- model team/player transitions and report wider uncertainty.

For evaluation, train through season *t* and predict season *t+1*, rolling this origin forward. Never train on future-season roster/coach information that was not known at the forecast cutoff.

## 12. Add coaching and contextual features

Join coaching tenure and tendencies by team and season/game date, using only information known at forecast time. Recommended features include:
- coach tenure year and change indicator;
- prior-game pass/run rate, pace, fourth-down aggressiveness;
- team/player snap share and usage trends;
- opponent defensive performance from prior games;
- rest days, travel, home/away, weather forecasts available at kickoff;
- pregame injury status and depth-chart role.

Scheme labels such as coverage family, blitz, motion, and personnel can be sparse or proprietary. Add only where sourced, documented, and available before kickoff.

In [13]:
# A reusable leakage checklist for feature engineering.
leakage_checklist = pd.DataFrame([
    ["Player outcome rolling stats", "Shift by one game before rolling", "Required"],
    ["Opponent defense", "Aggregate only prior completed games", "Required"],
    ["Coach/team tendencies", "Use tenure valid as of prediction cutoff", "Required"],
    ["Injuries", "Use pregame report timestamp, not postgame final status", "Required"],
    ["Weather", "Use forecast available before kickoff, not observed postgame weather", "Required"],
    ["Rest-of-season schedule", "Exclude games after cutoff from historical features", "Required"],
    ["Next-year roster/coach", "Do not use future facts not known at prediction time", "Required"],
    ["Random row split", "Avoid; use chronological or rolling-origin backtest", "Required"],
], columns=["feature_family","rule","status"])
display(leakage_checklist)
leakage_checklist.to_csv(OUTPUT_ROOT / "leakage_checklist.csv", index=False)

,feature_family,rule,status
0,Player outcome rolling stats,Shift by one game before rolling,Required
1,Opponent defense,Aggregate only prior completed games,Required
2,Coach/team tendencies,Use tenure valid as of prediction cutoff,Required
3,Injuries,"Use pregame report timestamp, not postgame fin...",Required
4,Weather,"Use forecast available before kickoff, not obs...",Required
5,Rest-of-season schedule,Exclude games after cutoff from historical fea...,Required
6,Next-year roster/coach,Do not use future facts not known at predictio...,Required
7,Random row split,Avoid; use chronological or rolling-origin bac...,Required


## 13. Final artifacts and model card

Save the data cutoff, target definition, feature list, train/validation/test seasons, metrics, and source versions with every model. Forecasts should include intervals and a warning when the model is extrapolating to rookies, new roles, or new coaching staffs.

In [14]:
run_card = {
    "created_at_utc": pd.Timestamp.utcnow().isoformat(),
    "data_root": str(DATA_ROOT),
    "target": TARGET,
    "feature_columns": feature_cols if "feature_cols" in globals() else [],
    "split": split_info if "split_info" in globals() else None,
    "available_tables": {name: {"rows": int(len(df)), "columns": int(len(df.columns))} for name, df in data.items()},
    "notes": [
        "Inspect and validate source schemas before interpreting results.",
        "Position trends are descriptive, not causal.",
        "Neural network metrics must be compared against simple baselines.",
        "Forecast performance is not guaranteed; use chronological backtests."
    ]
}
with open(OUTPUT_ROOT / "analysis_run_card.json","w",encoding="utf-8") as f:
    json.dump(run_card, f, indent=2, default=str)
print("Analysis artifacts written to", OUTPUT_ROOT.resolve())

Analysis artifacts written to C:\Users\Stephen\Desktop\Sports\FantasySports\NFL\nfl_analysis
